# Jinja for Classical NLP: Sentiment Analysis Prompt Templates

This notebook demonstrates a more relatable use case for Jinja: managing prompt templates for classical NLP tasks such as sentiment analysis.

The example is simple enough to understand quickly, but complex enough to show why a templating library is useful. We will render several prompt variants from the same data:

1. document-level three-class sentiment analysis,
2. document-level sentiment analysis with a `mixed` label,
3. aspect-based sentiment analysis,
4. batch sentiment annotation prompts.

The main lesson is that once prompts contain variants, conditional rules, few-shot examples, different label spaces, and different output schemas, plain Python strings become hard to maintain. Jinja lets us keep prompt text in external files and use clean template logic.


## 1. Install Jinja

Run this cell if Jinja is not already installed.


In [ ]:
# Uncomment if needed:
# %pip install jinja2

## 2. Create the mini project structure

In [ ]:
from pathlib import Path
import json
from collections import Counter

PROJECT_DIR = Path("nlp_jinja_sentiment_demo")
DATA_DIR = PROJECT_DIR / "data"
TEMPLATES_DIR = PROJECT_DIR / "templates"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

for directory in [DATA_DIR, TEMPLATES_DIR, OUTPUTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_DIR.resolve()


## 3. Create a small review dataset

The reviews are deliberately varied:

- some are clearly positive or negative,
- some are neutral,
- some contain mixed sentiment,
- one contains sarcasm,
- each review has aspects for aspect-based sentiment analysis.


In [ ]:
reviews = [
  {
    "id": "r001",
    "domain": "electronics",
    "text": "The battery life is excellent, but the keyboard feels cheap.",
    "language": "en",
    "aspects": [
      "battery",
      "keyboard",
      "overall_product"
    ],
    "gold_sentiment": "mixed",
    "contains_sarcasm": False
  },
  {
    "id": "r002",
    "domain": "restaurant",
    "text": "The food was lovely, but the service took forever.",
    "language": "en",
    "aspects": [
      "food",
      "service"
    ],
    "gold_sentiment": "mixed",
    "contains_sarcasm": False
  },
  {
    "id": "r003",
    "domain": "delivery",
    "text": "The package arrived two days early and everything was intact.",
    "language": "en",
    "aspects": [
      "delivery_time",
      "packaging"
    ],
    "gold_sentiment": "positive",
    "contains_sarcasm": False
  },
  {
    "id": "r004",
    "domain": "hotel",
    "text": "The room was clean. Nothing special, but nothing bad either.",
    "language": "en",
    "aspects": [
      "room_cleanliness",
      "overall_experience"
    ],
    "gold_sentiment": "neutral",
    "contains_sarcasm": False
  },
  {
    "id": "r005",
    "domain": "airline",
    "text": "I guess losing my suitcase is their idea of premium service.",
    "language": "en",
    "aspects": [
      "baggage_handling",
      "service"
    ],
    "gold_sentiment": "negative",
    "contains_sarcasm": True
  },
  {
    "id": "r006",
    "domain": "mobile_app",
    "text": "The latest update fixed the crashes, but the new interface is confusing.",
    "language": "en",
    "aspects": [
      "stability",
      "interface"
    ],
    "gold_sentiment": "mixed",
    "contains_sarcasm": False
  },
  {
    "id": "r007",
    "domain": "audio",
    "text": "Noise cancellation is excellent, and the headphones are comfortable for long calls.",
    "language": "en",
    "aspects": [
      "noise_cancellation",
      "comfort"
    ],
    "gold_sentiment": "positive",
    "contains_sarcasm": False
  },
  {
    "id": "r008",
    "domain": "banking",
    "text": "Support answered quickly but did not solve the issue.",
    "language": "en",
    "aspects": [
      "response_time",
      "problem_resolution"
    ],
    "gold_sentiment": "negative",
    "contains_sarcasm": False
  }
]

examples = [
  {
    "text": "The camera is sharp, but the battery dies too fast.",
    "label": "mixed",
    "reason": "It contains one positive and one negative opinion."
  },
  {
    "text": "The check-in was quick and the staff were friendly.",
    "label": "positive",
    "reason": "Both expressed opinions are favorable."
  },
  {
    "text": "It works exactly as described. No complaints.",
    "label": "positive",
    "reason": "The user expresses satisfaction."
  },
  {
    "text": "Great, another update that broke everything.",
    "label": "negative",
    "reason": "The wording is sarcastic; the intended sentiment is negative."
  },
  {
    "text": "The room was available.",
    "label": "neutral",
    "reason": "This is factual and does not clearly express an opinion."
  }
]

task_configs = {
  "project": {
    "name": "Relatable NLP Prompt Templating Demo",
    "description": "A tiny project showing how Jinja helps manage conditional prompt variants for sentiment-analysis tasks.",
    "default_language": "English"
  },
  "tasks": [
    {
      "name": "document_three_class",
      "mode": "document",
      "labels": [
        "positive",
        "neutral",
        "negative"
      ],
      "allow_mixed": False,
      "include_few_shot": True,
      "include_reasoning_field": True,
      "json_only": True,
      "max_examples": 3
    },
    {
      "name": "document_with_mixed",
      "mode": "document",
      "labels": [
        "positive",
        "neutral",
        "negative",
        "mixed"
      ],
      "allow_mixed": True,
      "include_few_shot": True,
      "include_reasoning_field": True,
      "json_only": True,
      "max_examples": 5
    },
    {
      "name": "aspect_based",
      "mode": "aspect",
      "labels": [
        "positive",
        "neutral",
        "negative",
        "mixed",
        "not_mentioned"
      ],
      "allow_mixed": True,
      "include_few_shot": False,
      "include_reasoning_field": False,
      "json_only": True,
      "max_examples": 0
    }
  ]
}

(DATA_DIR / "reviews.json").write_text(json.dumps(reviews, indent=2), encoding="utf-8")
(DATA_DIR / "few_shot_examples.json").write_text(json.dumps(examples, indent=2), encoding="utf-8")
(DATA_DIR / "task_configs.json").write_text(json.dumps(task_configs, indent=2), encoding="utf-8")

print(f"Wrote {len(reviews)} reviews and {len(task_configs['tasks'])} task configurations.")


## 4. Define reusable Jinja templates

Here we create three external templates:

- `sentiment_prompt.txt.j2` for a single review,
- `batch_prompt.txt.j2` for many reviews,
- `report.md.j2` for a project summary.

Notice that the templates contain `if` statements and `for` loops. This is the key reason to use Jinja instead of hard-coded Python strings.


In [ ]:
macros_template = "{% macro comma_labels(labels) -%}\n{%- for label in labels -%}\n`{{ label }}`{% if not loop.last %}, {% endif %}\n{%- endfor -%}\n{%- endmacro %}\n\n{% macro json_string_array(items) -%}\n[{{ items | map('tojson') | join(', ') }}]\n{%- endmacro %}\n"
sentiment_prompt_template = '{% import "macros.j2" as macros -%}\nYou are an expert NLP annotator performing sentiment analysis.\n\nProject: {{ project.name }}\nTask variant: {{ task.name }}\nTask mode: {{ task.mode }}\nReview ID: {{ review.id }}\nDomain: {{ review.domain }}\nLanguage: {{ review.language }}\n\nText:\n"{{ review.text }}"\n\nAllowed sentiment labels: {{ macros.comma_labels(task.labels) }}\n\n{% if task.mode == "document" -%}\nClassify the overall sentiment of the whole review.\n{% if task.allow_mixed -%}\nUse `mixed` when the review contains clearly positive and negative opinions of comparable importance.\n{% else -%}\nDo not use a mixed label. If sentiment is mixed, choose the dominant sentiment and lower the confidence.\n{% endif -%}\n{% elif task.mode == "aspect" -%}\nClassify sentiment separately for each aspect below:\n{% for aspect in review.aspects -%}\n- {{ aspect }}\n{% endfor -%}\nUse `not_mentioned` only if the review does not contain enough information about that aspect.\n{% endif %}\n\n{% if review.contains_sarcasm -%}\nImportant: this review may contain sarcasm. Interpret the intended sentiment, not only the literal wording.\n{% endif -%}\n\n{% if task.include_few_shot -%}\nExamples:\n{% for example in examples[:task.max_examples] %}\nInput: "{{ example.text }}"\nOutput:\n{\n  "sentiment": "{{ example.label }}",\n  "reason": "{{ example.reason }}"\n}\n{% endfor %}\n{% endif -%}\n\nReturn {% if task.json_only %}only valid JSON{% else %}a concise structured answer{% endif %}.\n\n{% if task.mode == "document" -%}\nRequired JSON schema:\n{\n  "review_id": "{{ review.id }}",\n  "sentiment": "one of: {{ task.labels | join(\', \') }}",\n  "confidence": "number between 0 and 1"{% if task.include_reasoning_field %},\n  "reason": "one short sentence explaining the decision"{% endif %}\n}\n{% elif task.mode == "aspect" -%}\nRequired JSON schema:\n{\n  "review_id": "{{ review.id }}",\n  "aspect_sentiments": [\n    {% for aspect in review.aspects -%}\n    {\n      "aspect": "{{ aspect }}",\n      "sentiment": "one of: {{ task.labels | join(\', \') }}",\n      "evidence": "short phrase from the review"\n    }{% if not loop.last %},{% endif %}\n    {% endfor -%}\n  ]\n}\n{% endif -%}\n'
batch_prompt_template = '{% import "macros.j2" as macros -%}\nYou are an expert NLP annotator. Annotate a batch of customer reviews.\n\nTask variant: {{ task.name }}\nMode: {{ task.mode }}\nAllowed labels: {{ macros.comma_labels(task.labels) }}\n\n{% if task.mode == "document" -%}\nFor each review, return one overall sentiment label.\n{% if task.allow_mixed -%}\nUse `mixed` when positive and negative opinions are both important.\n{% endif -%}\n{% elif task.mode == "aspect" -%}\nFor each review, return sentiment for each listed aspect.\n{% endif %}\n\nReviews:\n{% for review in reviews %}\n{{ loop.index }}. ID: {{ review.id }}\n   Domain: {{ review.domain }}\n   Text: "{{ review.text }}"\n   {% if task.mode == "aspect" -%}\n   Aspects: {{ review.aspects | join(", ") }}\n   {% endif -%}\n   {% if review.contains_sarcasm -%}\n   Note: possible sarcasm.\n   {% endif %}\n{% endfor %}\n\nReturn only valid JSON with one item per review.\n'
report_template = '# {{ project.name }}\n\n{{ project.description }}\n\n## Dataset Summary\n\nTotal reviews: {{ reviews | length }}\n\n{% set sarcastic = reviews | selectattr("contains_sarcasm") | list -%}\nReviews with possible sarcasm: {{ sarcastic | length }}\n\nGold-label distribution:\n\n{% for label, count in gold_distribution.items() -%}\n- {{ label }}: {{ count }}\n{% endfor %}\n\n## Prompt Variants\n\n{% for task in tasks %}\n### {{ loop.index }}. {{ task.name }}\n\n- Mode: `{{ task.mode }}`\n- Labels: {{ task.labels | join(", ") }}\n- Few-shot examples: {% if task.include_few_shot %}yes, up to {{ task.max_examples }}{% else %}no{% endif %}\n- JSON-only response: {% if task.json_only %}yes{% else %}no{% endif %}\n\n{% if task.mode == "aspect" %}\nThis variant changes the output schema from one document-level label to a list of aspect-level labels.\n{% elif task.allow_mixed %}\nThis variant allows a `mixed` sentiment class, which is useful for reviews with both praise and criticism.\n{% else %}\nThis variant forces a single three-class document-level decision.\n{% endif %}\n\n{% endfor %}\n\n## Why Jinja Is Useful Here\n\nThe Python code does not need separate long strings for every NLP prompt variant.  \nThe templates handle:\n\n- conditional task modes,\n- optional few-shot examples,\n- different label spaces,\n- sarcasm warnings,\n- document-level versus aspect-level JSON schemas,\n- batch versus single-review prompts.\n'

(TEMPLATES_DIR / "macros.j2").write_text(macros_template, encoding="utf-8")
(TEMPLATES_DIR / "sentiment_prompt.txt.j2").write_text(sentiment_prompt_template, encoding="utf-8")
(TEMPLATES_DIR / "batch_prompt.txt.j2").write_text(batch_prompt_template, encoding="utf-8")
(TEMPLATES_DIR / "report.md.j2").write_text(report_template, encoding="utf-8")

print("Templates written.")


## 5. Render single-review and batch prompts

The rendering code stays short. Most of the conditional wording lives in the templates.


In [ ]:
from jinja2 import Environment, FileSystemLoader, StrictUndefined

env = Environment(
    loader=FileSystemLoader(TEMPLATES_DIR),
    undefined=StrictUndefined,
    trim_blocks=True,
    lstrip_blocks=True,
)

single_prompt = env.get_template("sentiment_prompt.txt.j2")
batch_prompt = env.get_template("batch_prompt.txt.j2")
report = env.get_template("report.md.j2")

project = task_configs["project"]
tasks = task_configs["tasks"]

for task in tasks:
    task_output_dir = OUTPUTS_DIR / task["name"]
    task_output_dir.mkdir(parents=True, exist_ok=True)

    for review in reviews:
        rendered = single_prompt.render(
            project=project,
            task=task,
            review=review,
            examples=examples,
        )
        (task_output_dir / f"{review['id']}.txt").write_text(rendered, encoding="utf-8")

    rendered_batch = batch_prompt.render(task=task, reviews=reviews)
    (task_output_dir / "batch_prompt.txt").write_text(rendered_batch, encoding="utf-8")

gold_distribution = dict(Counter(review["gold_sentiment"] for review in reviews))
rendered_report = report.render(
    project=project,
    reviews=reviews,
    tasks=tasks,
    gold_distribution=gold_distribution,
)
(OUTPUTS_DIR / "project_report.md").write_text(rendered_report, encoding="utf-8")

print("Generated files:")
for path in sorted(OUTPUTS_DIR.rglob("*")):
    if path.is_file():
        print("-", path)


## 6. Inspect a document-level prompt

This variant uses a simple document-level sentiment task with labels `positive`, `neutral`, and `negative`. It also includes few-shot examples.


In [ ]:
print((OUTPUTS_DIR / "document_three_class" / "r005.txt").read_text(encoding="utf-8"))

## 7. Inspect a variant that allows `mixed`

The same review and template can produce different instructions when the task configuration changes. Here, `mixed` becomes an allowed label.


In [ ]:
print((OUTPUTS_DIR / "document_with_mixed" / "r001.txt").read_text(encoding="utf-8"))

## 8. Inspect an aspect-based sentiment prompt

This variant changes the output schema completely: instead of one label for the whole review, it asks for sentiment per aspect.


In [ ]:
print((OUTPUTS_DIR / "aspect_based" / "r002.txt").read_text(encoding="utf-8"))

## 9. Inspect a batch prompt

Batch prompts are a natural use case for templating because the prompt loops over many records while preserving consistent instructions.


In [ ]:
print((OUTPUTS_DIR / "aspect_based" / "batch_prompt.txt").read_text(encoding="utf-8"))

## 10. Render a small project report

Jinja is not only useful for LLM prompts. The same pattern works for reports, dataset cards, annotation guidelines, and evaluation summaries.


In [ ]:
from IPython.display import Markdown, display

display(Markdown((OUTPUTS_DIR / "project_report.md").read_text(encoding="utf-8")))


## 11. Why this is better than plain Python strings

For a tiny one-off prompt, an f-string is fine. But once the project has multiple prompt variants, Jinja becomes useful because it separates:

- **data**: reviews, labels, aspects, task configuration,
- **logic**: conditional branches and loops in templates,
- **presentation**: wording of prompts and reports,
- **execution**: small Python rendering code.

This makes the project easier to maintain, especially when NLP tasks evolve from simple classification to few-shot, schema-constrained, batch, or aspect-based workflows.
